In [14]:

import csv
import qrcode
import random
import shutil
from pathlib import Path


In [15]:

CHARS = "ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjkmnpqrstuvwxyz23456789"
## 去除了0 1 O I L


In [17]:

def generate_code(groups=4, group_len=4):
    parts = []
    for _ in range(groups):
        part = ''.join(random.choice(CHARS) for _ in range(group_len))
        parts.append(part)
    return "-".join(parts)


count = 300
csv_file = Path("ticket_codes.csv")
output_dir = Path("qrcodes")


In [23]:

# 1. 删除旧 CSV
if csv_file.exists():
    csv_file.unlink()

# 2. 删除旧二维码文件夹及其中100张图
if output_dir.exists():
    shutil.rmtree(output_dir)

# 3. 重新创建二维码文件夹
output_dir.mkdir(exist_ok=True)

# 4. 重新生成100个兑换码
codes = set()

while len(codes) < count:
    codes.add(generate_code(groups=4, group_len=4))
    # groups=4 表示4组，group_len=4 表示每组4位
    # 例如：K7QW-9M2X-TR6P-B8ND

# 5. 重新生成新的 CSV 和新的二维码
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ticket_no", "code", "qr_file"])

    for i, code in enumerate(sorted(codes), start=1):
        ticket_no = f"A{i:04d}"

        qr = qrcode.QRCode(
            version=None,
            error_correction=qrcode.constants.ERROR_CORRECT_H,
            box_size=12,
            border=4
            )
        
        qr.add_data(code)
        qr.make(fit=True)

        img = qr.make_image(fill_color="black", back_color="white")
        filename = output_dir / f"{code}_{ticket_no}.png"
        img.save(filename)

        writer.writerow([ticket_no, code, str(filename)])

In [25]:
# ===== 新增：合成票根图片（编号 + 二维码 + 兑换码） =====

import csv
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont


In [26]:

# 配置
CSV_FILE = Path("ticket_codes.csv")          # 已有的CSV
QR_DIR = Path("qrcodes")                     # 已有的二维码文件夹
OUTPUT_DIR = Path("ticket_stubs")            # 合成图片输出文件夹
QR_TARGET_SIZE = (300, 300)                  # 二维码在票根上的显示尺寸（像素）
PADDING = 20                                 # 内边距
LINE_SPACING = 10                            # 行间距

# 创建输出目录
OUTPUT_DIR.mkdir(exist_ok=True)


In [28]:
#
#  尝试加载一个稍微好看的字体（若系统中没有，则使用默认字体）
try:
    # 这里可以替换为你系统里已有的中英文字体路径，例如 "arial.ttf" 或 "simhei.ttf"
    font_title = ImageFont.truetype("arial.ttf", 28)
    font_code = ImageFont.truetype("arial.ttf", 22)
except IOError:
    font_title = ImageFont.load_default()
    font_code = ImageFont.load_default()


In [31]:
# 读取CSV，逐行合成
with open(CSV_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        ticket_no = row["ticket_no"]
        code = row["code"]
        qr_path = Path(row["qr_file"])

        # 1. 打开二维码图片并调整大小
        qr_img = Image.open(qr_path).resize(QR_TARGET_SIZE, Image.Resampling.LANCZOS)

        # 2. 计算票根画布尺寸
        #    画布宽度 = 二维码宽度 + 左右内边距
        #    画布高度 = 上边距 + 标题高度 + 间距 + 二维码高度 + 间距 + 兑换码高度 + 下边距
        title_text = f"No. {ticket_no}"
        # 估算文字尺寸（默认字体无法准确测宽，我们采用固定宽度；若用truetype可精确测量）
        # 这里为了简化，直接给一个经验值，您可调整
        title_height = 40
        code_height = 40
        margin_top = 30
        margin_bottom = 30
        width = QR_TARGET_SIZE[0] + 2 * PADDING
        height = margin_top + title_height + LINE_SPACING + QR_TARGET_SIZE[1] + LINE_SPACING + code_height + margin_bottom

        # 3. 创建白色背景画布
        canvas = Image.new("RGB", (width, height), "white")
        draw = ImageDraw.Draw(canvas)

        # 4. 绘制票号（居中）
        title_bbox = draw.textbbox((0, 0), title_text, font=font_title)
        title_width = title_bbox[2] - title_bbox[0]
        title_x = (width - title_width) // 2
        title_y = margin_top
        draw.text((title_x, title_y), title_text, fill="black", font=font_title)

        # 5. 粘贴二维码（居中）
        qr_x = (width - QR_TARGET_SIZE[0]) // 2
        qr_y = margin_top + title_height + LINE_SPACING
        canvas.paste(qr_img, (qr_x, qr_y))

        # 6. 绘制兑换码（居中）
        code_text = f"Code: {code}"
        code_bbox = draw.textbbox((0, 0), code_text, font=font_code)
        code_width = code_bbox[2] - code_bbox[0]
        code_x = (width - code_width) // 2
        code_y = margin_top + title_height + LINE_SPACING + QR_TARGET_SIZE[1] + LINE_SPACING
        draw.text((code_x, code_y), code_text, fill="black", font=font_code)

        # 7. 保存合成图片
        out_filename = OUTPUT_DIR / f"{ticket_no}_stub.png"
        canvas.save(out_filename)
        print(f"已生成: {out_filename}")

print("所有票根合成完毕！")

已生成: ticket_stubs\A0001_stub.png
已生成: ticket_stubs\A0002_stub.png
已生成: ticket_stubs\A0003_stub.png
已生成: ticket_stubs\A0004_stub.png
已生成: ticket_stubs\A0005_stub.png
已生成: ticket_stubs\A0006_stub.png
已生成: ticket_stubs\A0007_stub.png
已生成: ticket_stubs\A0008_stub.png
已生成: ticket_stubs\A0009_stub.png
已生成: ticket_stubs\A0010_stub.png
已生成: ticket_stubs\A0011_stub.png
已生成: ticket_stubs\A0012_stub.png
已生成: ticket_stubs\A0013_stub.png
已生成: ticket_stubs\A0014_stub.png
已生成: ticket_stubs\A0015_stub.png
已生成: ticket_stubs\A0016_stub.png
已生成: ticket_stubs\A0017_stub.png
已生成: ticket_stubs\A0018_stub.png
已生成: ticket_stubs\A0019_stub.png
已生成: ticket_stubs\A0020_stub.png
已生成: ticket_stubs\A0021_stub.png
已生成: ticket_stubs\A0022_stub.png
已生成: ticket_stubs\A0023_stub.png
已生成: ticket_stubs\A0024_stub.png
已生成: ticket_stubs\A0025_stub.png
已生成: ticket_stubs\A0026_stub.png
已生成: ticket_stubs\A0027_stub.png
已生成: ticket_stubs\A0028_stub.png
已生成: ticket_stubs\A0029_stub.png
已生成: ticket_stubs\A0030_stub.png
已生成: ticke